In [ ]:
#| default_exp meta_learning.environments.wrappers

In [ ]:
#| export
import gym
import numpy as np
from gym import spaces
from gym.envs.registration import load
import random
import gc

import torch
import torch.nn.functional as F

In [ ]:
#| export

class VariBadWrapper(gym.Wrapper):
    def __init__(self,
                 env,
                 episodes_per_task,
                 env_type,
                 args
                 ):
        """
        Wrapper, creates a multi-episode (BA)MDP around a one-episode MDP. Automatically deals with
        - horizons H in the MDP vs horizons H+ in the BAMDP,
        - resetting the tasks
        - adding the done info to the state (might be needed to make states markov)
        """

        super().__init__(env)

        self.env_type = env_type
        self.args = args

        # make sure we can call these attributes even if the orig env does not have them
        if not hasattr(self.env.unwrapped, 'task_dim'):
            self.env.unwrapped.task_dim = 0
        if not hasattr(self.env.unwrapped, 'belief_dim'):
            self.env.unwrapped.belief_dim = 0
        if not hasattr(self.env.unwrapped, 'get_belief'):
            self.env.unwrapped.get_belief = lambda: None
        if not hasattr(self.env.unwrapped, 'num_states'):
            self.env.unwrapped.num_states = None
        if not hasattr(self.env.unwrapped, '_max_episode_steps'):  # Meta-World ML10/ML45
            self.env.unwrapped._max_episode_steps = env.horizon

        if episodes_per_task > 1:
            self.add_done_info = True
        else:
            self.add_done_info = False

        if self.add_done_info:
            if isinstance(self.observation_space, spaces.Box) or isinstance(self.observation_space,
                                                                            rand_param_envs.gym.spaces.box.Box):
                if len(self.observation_space.shape) > 1:
                    raise ValueError  # can't add additional info for obs of more than 1D
                self.observation_space = spaces.Box(low=np.array([*self.observation_space.low, 0]),
                                                    # shape will be deduced from this
                                                    high=np.array([*self.observation_space.high, 1])
                                                    )
            else:
                # Not implemented. Would need to add something simliar for the other possible spaces,
                # "Space", "Discrete", "MultiDiscrete", "MultiBinary", "Tuple", "Dict", "flatdim", "flatten", "unflatten"
                raise NotImplementedError

        # calculate horizon length H^+
        self.episodes_per_task = episodes_per_task
        # counts the number of episodes
        self.episode_count = 0

        # count timesteps in BAMDP
        self.step_count_bamdp = 0.0
        # the horizon in the BAMDP is the one in the MDP times the number of episodes per task,
        # and if we train a policy that maximises the return over all episodes
        # we add transitions to the reset start in-between episodes
        try:
            self.horizon_bamdp = self.episodes_per_task * self.env._max_episode_steps
        except AttributeError:
            self.horizon_bamdp = self.episodes_per_task * self.env.unwrapped._max_episode_steps

        # add dummy timesteps in-between episodes for resetting the MDP
        self.horizon_bamdp += self.episodes_per_task - 1

        # this tells us if we have reached the horizon in the underlying MDP
        self.done_mdp = True

    def reset(self, task=None):
        """ Resets the BAMDP """

        try:  # meta-world cannot take task spec
            self.env.reset_task(task)
        except TypeError:
            self.env.reset_task()

        # normal reset
        try:
            state = self.env.reset()
        except AttributeError:
            state = self.env.unwrapped.reset()

        self.episode_count = 0
        self.step_count_bamdp = 0
        self.done_mdp = False
        if self.add_done_info:
            state = np.concatenate((state, [0.0]))

        return state

    def reset_mdp(self):
        """ Resets the underlying MDP only (*not* the task). """
        state = self.env.reset()
        if self.add_done_info:
            state = np.concatenate((state, [0.0]))
        self.done_mdp = False
        return state

    def step(self, action):

        # do normal environment step in MDP
        state, reward, self.done_mdp, info = self.env.step(action)

        if self.env_type == 'metaworld':
            if self.env._max_episode_steps == self.env.curr_path_length:
                self.done_mdp = True
                info['bad_transition'] = True

        info['done_mdp'] = self.done_mdp

        if self.add_done_info:
            state = np.concatenate((state, [float(self.done_mdp)]))

        self.step_count_bamdp += 1
        # if we want to maximise performance over multiple episodes,
        # only say "done" when we collected enough episodes in this task
        done_bamdp = False
        if self.done_mdp:
            self.episode_count += 1
            if self.episode_count == self.episodes_per_task:
                done_bamdp = True

        if self.done_mdp and not done_bamdp:
            if self.env_type == 'Maze':
                # In minecraft and tmaze envs, it is necessary to see start state of next ep, but not terminal state
                info['term_state'] = state
                state = self.reset_mdp()
                if self.add_done_info:
                    state[-1] = 1.0
                info['start_state'] = state
            else:
                info['start_state'] = self.reset_mdp()

        return state, reward, done_bamdp, info

    def __getattr__(self, attr):
        """
        If env does not have the attribute then call the attribute in the wrapped_env
        (This one's only needed for mujoco 131)
        """
        try:
            orig_attr = self.__getattribute__(attr)
        except AttributeError:
            try:
                orig_attr = self.env.__getattribute__(attr)
            except AttributeError:
                orig_attr = self.unwrapped.__getattribute__(attr)
        if callable(orig_attr):
            def hooked(*args, **kwargs):
                result = orig_attr(*args, **kwargs)
                return result

            return hooked
        else:
            return orig_attr

In [ ]:
#| export
class TimeLimitMask(gym.Wrapper):

    def step(self, action):
        obs, rew, done, info = self.env.step(action)
        if done and self.env._max_episode_steps == self.env._elapsed_steps:
            info['bad_transition'] = True
        return obs, rew, done, info

    def reset(self, **kwargs):
        return self.env.reset(**kwargs)

    def __getattr__(self, attr):
        """
        If env does not have the attribute then call the attribute in the wrapped_env
        (This one's only needed for mujoco 131)
        """
        try:
            orig_attr = self.__getattribute__(attr)
        except AttributeError:
            try:
                orig_attr = self.env.__getattribute__(attr)
            except AttributeError:
                orig_attr = self.unwrapped.__getattribute__(attr)
        if callable(orig_attr):
            def hooked(*args, **kwargs):
                result = orig_attr(*args, **kwargs)
                return result

            return hooked
        else:
            return orig_attr

In [ ]:
#| export

class PrevActRewWrapper(gym.Wrapper):
    """
    Concatenate (prev_action, prev_reward) to each observation
    so that a recurrent policy sees (s_{t-1}, a_{t-1}, r_{t-1}).

    The first observation after `reset()` contains zeros for the
    action & reward slots.
    """

    def __init__(self, env: gym.Env):
        super().__init__(env)

        # --- build new observation space ------------------------------------
        obs_low,  obs_high  = env.observation_space.low,  env.observation_space.high
        act_low,  act_high  = env.action_space.low,       env.action_space.high
        rew_low,  rew_high  = np.array([-np.inf], dtype=np.float32), np.array([np.inf], dtype=np.float32)

        self.observation_space = gym.spaces.Box(
            low=np.concatenate([obs_low,  act_low,  rew_low]),
            high=np.concatenate([obs_high, act_high, rew_high]),
            dtype=np.float32
        )

        # buffer for previous step
        self._prev_action = np.zeros(env.action_space.shape, dtype=np.float32)
        self._prev_reward = 0.0

    # --------------------------------------------------------------------- #
    def reset(self, **kwargs):
        obs = self.env.reset(**kwargs)
        self._prev_action.fill(0.0)
        self._prev_reward = 0.0
        return self._augment(obs)

    # --------------------------------------------------------------------- #
    def step(self, action):
        obs, reward, done, info = self.env.step(action)

        # augment *next* observation with the action & reward *just taken*
        aug_obs = self._augment(obs, action, reward)

        # store for the following step
        self._prev_action = np.asarray(action, dtype=np.float32).copy()
        self._prev_reward = float(reward)

        return aug_obs, reward, done, info

    # --------------------------------------------------------------------- #
    # helper
    def _augment(self, obs, act=None, rew=None):
        """
        Build concatenated observation.  If `act` / `rew` are `None`
        use the stored values from the previous time-step.
        """
        if act is None:
            act = self._prev_action
        if rew is None:
            rew = self._prev_reward
        return np.concatenate([obs, act, [rew]]).astype(np.float32)


In [ ]:
from ddopai.meta_learning.environments.pricing_env.pricing_env import PricingEnv

# Create a test environment
test_env = PricingEnv(p_low=0, p_high=5, 
                      nb_features=5, horizon_choices=[500],
                      mean_alpha=1.2, std_alpha=0.2,
                      mean_beta=-0.3, std_beta=0.2,
                      noise_std_choices=[0.2], 
                      inv_ratio_mean=0.0, 
                      inv_ratio_std=0.0)


In [ ]:
wrapped_env = PrevActRewWrapper(test_env)

In [ ]:
wrapped_env.step(np.array([2]))

(array([0.        , 0.3990949 , 0.08974466, 0.08948473, 0.12865682,
        0.3609438 , 2.        , 2.1665137 ], dtype=float32),
 2.1665137081816908,
 False,
 {'task': array([ 1.1938479e+00,  1.5092746e+00,  1.1094871e+00,  1.1007974e+00,
          1.4028376e+00, -1.7606144e-01, -2.4605061e-01, -4.1061407e-01,
         -3.2327956e-01, -8.7723754e-02,  2.0000000e-01,  0.0000000e+00,
          5.0000000e+02], dtype=float32),
  'noise': -0.18115256878360528,
  'demand': 1.0832568540908454,
  'sales': 1.0832568540908454,
  'inv': 0.0})